# 🏏 YOLOv8 - Cricket Player Detection Training
Complete ready-to-run notebook for training YOLOv8 on a Roboflow-exported
Cricket Player Detection dataset stored in Google Drive.

**Dataset structure expected (Roboflow export):**
```
dataset_root/
├── data.yaml
├── README.dataset.txt
├── README.roboflow.txt
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

Just set `DATASET_DIR` in the config cell below and run all cells top to bottom.
Training runs for **100 epochs**, checkpoints/results are saved to Drive so you
don't lose progress on a Colab runtime reset.

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Install Ultralytics (YOLOv8)
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 4. Configure dataset path
Set `DATASET_DIR` to the folder in your Drive that contains `data.yaml`,
`train/`, `valid/`, `test/` (the folder shown in your screenshot).

Example: if the folder is at
`My Drive/Cricket Player Detection/Cricket-Player-Detection-1`, set:

```python
DATASET_DIR = "/content/drive/MyDrive/Cricket Player Detection/Cricket-Player-Detection-1"
```

In [ ]:
# 4. Set your dataset folder path (EDIT THIS)
DATASET_DIR = "/content/drive/MyDrive/Cricket Player Detection/Cricket-Player-Detection-1"  # <-- change if needed

# Where to save training runs/checkpoints in Drive (survives Colab resets)
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Cricket_Player_Detection_Training"

import os
assert os.path.isdir(DATASET_DIR), f"DATASET_DIR does not exist: {DATASET_DIR}"
print("Contents of DATASET_DIR:")
for f in sorted(os.listdir(DATASET_DIR)):
    print(" -", f)

In [ ]:
# 5. Verify train/valid/test images+labels subfolders exist
required = ["train/images", "train/labels", "valid/images", "valid/labels", "test/images", "test/labels"]
for r in required:
    p = os.path.join(DATASET_DIR, r)
    status = "OK" if os.path.isdir(p) else "MISSING"
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"{status:8s} {r:20s} ({n} files)")

In [ ]:
# 6. Fix data.yaml so paths are ABSOLUTE and correct for this Colab session
# (Roboflow-exported data.yaml often has relative paths like "../train/images"
#  which break once the folder is moved into Drive / Colab.)
import yaml

data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")
with open(data_yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

print("Original data.yaml:")
print(data_cfg)

data_cfg["path"]  = DATASET_DIR
data_cfg["train"] = "train/images"
data_cfg["val"]   = "valid/images"
data_cfg["test"]  = "test/images"

fixed_yaml_path = "/content/data.yaml"  # local copy Colab will actually use
with open(fixed_yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("\nFixed data.yaml written to:", fixed_yaml_path)
print(data_cfg)
print("\nClasses:", data_cfg.get("names"))
print("Number of classes (nc):", data_cfg.get("nc", len(data_cfg.get("names", []))))

## 7. Train YOLOv8
Trains for **100 epochs**. Checkpoints are saved directly into Drive
(`DRIVE_PROJECT_DIR`) so if the Colab runtime disconnects/resets, you can
resume from `last.pt` instead of starting over.

In [ ]:
# 7. Train
from ultralytics import YOLO

os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)

RUN_NAME = "cricket_player_yolov8"
MODEL_SIZE = "yolov8n.pt"   # options: yolov8n.pt (fastest) / yolov8s.pt / yolov8m.pt / yolov8l.pt / yolov8x.pt (most accurate)
EPOCHS = 100
IMG_SIZE = 640
BATCH = 16

model = YOLO(MODEL_SIZE)

results = model.train(
    data=fixed_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=DRIVE_PROJECT_DIR,   # saved straight to Drive
    name=RUN_NAME,
    exist_ok=True,
    patience=50,                 # early stopping patience
    save=True,
    save_period=10,               # extra checkpoint every 10 epochs
    device=0,                     # GPU
)

### Resuming after a Colab runtime reset
If training gets interrupted, re-run cells 1-6 above, then run this instead
of the training cell above (uses the last saved checkpoint):

```python
resume_model = YOLO(f"{DRIVE_PROJECT_DIR}/{RUN_NAME}/weights/last.pt")
results = resume_model.train(resume=True)
```

In [ ]:
# 8. Validate the trained model on the validation set
best_weights = f"{DRIVE_PROJECT_DIR}/{RUN_NAME}/weights/best.pt"
val_model = YOLO(best_weights)
metrics = val_model.val(data=fixed_yaml_path, split="val")

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

In [ ]:
# 9. Run inference on a few test images and show results
import glob
from IPython.display import Image as IPImage, display

test_images_dir = os.path.join(DATASET_DIR, "test", "images")
sample_images = sorted(glob.glob(os.path.join(test_images_dir, "*")))[:5]

pred_results = val_model.predict(
    source=sample_images,
    conf=0.25,
    save=True,
    project=DRIVE_PROJECT_DIR,
    name=f"{RUN_NAME}_predictions",
    exist_ok=True,
)

pred_dir = f"{DRIVE_PROJECT_DIR}/{RUN_NAME}_predictions"
for img_path in sorted(glob.glob(os.path.join(pred_dir, "*.jpg")))[:5]:
    display(IPImage(filename=img_path, width=500))

In [ ]:
# 10. Export trained model (optional formats)
val_model.export(format="onnx")   # creates best.onnx next to best.pt
# Other options: format="torchscript", format="tflite", format="engine" (TensorRT), etc.

## Done ✅
Your trained weights are saved permanently in Google Drive at:

```
{DRIVE_PROJECT_DIR}/{RUN_NAME}/weights/best.pt   # best checkpoint
{DRIVE_PROJECT_DIR}/{RUN_NAME}/weights/last.pt   # latest checkpoint (for resuming)
```

Training curves, confusion matrix, and sample predictions are also saved
under `{DRIVE_PROJECT_DIR}/{RUN_NAME}/`.